In [1]:
from getpass import getpass
token = getpass("Paste GitHub token: ")

!git clone https://{token}@github.com/Kairos-025/llm-qa-project.git
%cd llm-qa-project
!pip install -q -r requirements.txt

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

Cloning into 'llm-qa-project'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 187 (delta 93), reused 129 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (187/187), 44.86 MiB | 18.11 MiB/s, done.
Resolving deltas: 100% (93/93), done.
/content/llm-qa-project
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
GPU: Tesla T4


In [2]:
!python -m src.data_prep --test_size 1000

README.md: 100% 8.92k/8.92k [00:00<00:00, 21.4MB/s]

squad_v2/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/16.4M [00:00<?, ?B/s]s]
squad_v2/train-00000-of-00001.parquet: downloading bytes: 100% 15.9M/15.9M [00:00<00:00, 27.2MB/s, 1.57MB/s  ]
squad_v2/train-00000-of-00001.parquet: reconstructing file: 100% 16.4M/16.4M [00:00<00:00, 27.9MB/s, 1.62MB/s  ]

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:   0% 0.00/1.35M [00:00<?, ?B/s]s]
squad_v2/validation-00000-of-00001.parqu(…): downloading bytes: 100% 1.30M/1.30M [00:00<00:00, 2.85MB/s,  128kB/s  ]
squad_v2/validation-00000-of-00001.parqu(…): reconstructing file: 100% 1.35M/1.35M [00:00<00:00, 2.96MB/s,  134kB/s  ]
Generating train split: 100% 130319/130319 [00:00<00:00, 471549.87 examples/s]
Generating validation split: 100% 11873/11873 [00:00<00:00, 306102.95 examples/s]
wrote   1000 rows -> data/test.jsonl
wrote 130319 rows -> data/train.jsonl
wrote     20 rows -> data/fewshot_pool.jsonl
wrote   1204 ro

In [3]:
import torch, transformers, datasets, peft, trl, accelerate, bitsandbytes, sentence_transformers, faiss
for m in [torch, transformers, datasets, peft, trl, accelerate, bitsandbytes, sentence_transformers, faiss]:
    print(m.__name__, m.__version__)

torch 2.11.0+cu128
transformers 5.16.1
datasets 4.8.5
peft 0.20.0
trl 1.14.0
accelerate 1.14.0
bitsandbytes 0.50.2
sentence_transformers 5.7.0
faiss 1.15.1


In [4]:
import json, collections, statistics
from datasets import load_dataset

ds = load_dataset("rajpurkar/squad_v2")
test = [json.loads(l) for l in open("data/test.jsonl")]

def summary(name, rows_q, rows_ans):
    n = len(rows_q)
    unans = sum(1 for a in rows_ans if len(a) == 0)
    print(f"{name:<11} {n:>7} questions | {unans/n:.1%} unanswerable")

print("=== Split sizes ===")
for split in ["train", "validation"]:
    d = ds[split]
    summary(split, d["question"], [a["text"] for a in d["answers"]])
summary("our test", [r["question"] for r in test], [r["answers"] for r in test])

val = ds["validation"]
ans_words = [len(a["text"][0].split()) for a in val["answers"] if a["text"]]
ctx_words = [len(c.split()) for c in set(val["context"])]
print("\n=== Answer length (words, validation) ===")
print(f"mean {statistics.mean(ans_words):.1f} | median {statistics.median(ans_words)} | "
      f"95% are <= {sorted(ans_words)[int(.95*len(ans_words))]}")
print("\n=== Passage length (words, validation) ===")
print(f"mean {statistics.mean(ctx_words):.0f} | longest {max(ctx_words)}")
print(f"articles: {len(set(val['title']))} | unique passages: {len(ctx_words)}")

print("\n=== Question types (validation) ===")
first = collections.Counter(q.split()[0].lower() for q in val["question"])
for word, c in first.most_common(8):
    print(f"{word:<8} {c/len(val):.1%}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


=== Split sizes ===
train        130319 questions | 33.4% unanswerable
validation    11873 questions | 50.1% unanswerable
our test       1000 questions | 50.7% unanswerable

=== Answer length (words, validation) ===
mean 3.2 | median 2.0 | 95% are <= 10

=== Passage length (words, validation) ===
mean 127 | longest 629
articles: 35 | unique passages: 1204

=== Question types (validation) ===
what     49.0%
how      9.5%
who      8.3%
when     6.2%
in       4.3%
where    3.6%
which    2.8%
the      2.5%
